# Numerički integratori i grafovske neuronske mreže za simulaciju dinamike čestica
**Projekat: NANS (Numerički Algoritmi i Numerički Softver)**  
**Platforma/Prezentacija: Interaktivni Python Notebook**

Ova prezentacija predstavlja projekat za ocenu na kursu NANS, a funkcioniše i kao interaktivni uvod u projekat namenjen za GitHub repozitorijum.

## 1. Motivacija i Uvod

Simulacija dinamike čestica predstavlja ključno područje u računarskoj fizici. Tradicionalno se koriste **numerički integratori** kako bi se rešile diferencijalne jednačine kretanja iterativno kroz vreme. Ovi algoritmi su fizički utemeljeni, ali pate od velike osetljivosti na vremenski korak (`dt`) i izuzetno su računarski zahtevni.

Glavni cilj projekta je upoređivanje klasičnog numeričkog pristupa sa pristupom mašinskog učenja pomoću **Grafovskih Neuronskih Mreža (GNN)**, koje omogućavaju bržu simulaciju kompleksne fizike na osnovu prethodnog učenja interakcija iz generisanih podataka.


## 2. NANS u Fokusu: Numerička Simulacija (WCSPH)

Kao osnovu projekta i primarni izvor fizički potpuno korektnih podataka, razvili smo sopstveni **WCSPH (Weakly Compressible Smoothed Particle Hydrodynamics)** softver, umesto oslanjanja samo na jednostavne elastične sile na jednoj dimenziji prostora.

### Matematički Model
Dinamika tečnosti aproksimira se metodom čestica čije interakcije definiše SPH kernel funkcija $W(r, h)$, gde je $h$ radijus uticaja. Centralne jednačine obuhvataju:

1. **Gustina:** $\rho_i = \sum_j m_j W(r_{ij}, h)$  
2. **Pritisak fluida (Taitova jednačina stišljive tečnosti):** $P_i = k \left( \left(\frac{\rho_i}{\rho_0}\right)^\gamma - 1 \right)$
3. **Sila pritiska (gradijent pritiska SPH):** $\mathbf{F}_i^{press} = -\sum_j m_j \left( \frac{P_i}{\rho_i^2} + \frac{P_j}{\rho_j^2} \right) \nabla W(r_{ij}, h)$
4. **Sila viskoznosti:** $\mathbf{F}_i^{visc} = \mu \sum_j m_j \frac{\mathbf{v}_j - \mathbf{v}_i}{\rho_j} \nabla^2 W(r_{ij}, h)$

### Numerički Integrator u Simulaciji
Ključno za veran proračun numeričke simulacije iz projektnog zadatka je izbor efikasnog numeričkog integratora. Iako su podržani **Forward Euler** i **Runge-Kutta 4 (RK4)**, za neophodnu energetsku stabilnost WCSPH sistema implementirali smo varijantu **Symplectic Euler** integratora (koji je uspešno i računarski lakše zamenio projektom inicijalno predloženi Velocity Verlet metod):

$$ \mathbf{v}_{t+1} = \mathbf{v}_t + \mathbf{a}_t \Delta t $$
$$ \mathbf{x}_{t+1} = \mathbf{x}_t + \mathbf{v}_{t+1} \Delta t $$

Simplektički Euler obezbeđuje očuvanje energije faznog prostora duž veoma dugačkih trajektorija simulacije fluida uz izrazito niske računske zahteve.


In [ ]:
# Učitavamo okruženje i relevantne simulacione biblioteke.
import sys
import os
import torch
import numpy as np
import yaml
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# Usmeravanje ka korenskom direktorijumu ukoliko je pokrenuto iz presentacionog
if os.getcwd().split('/')[-1] == 'presentation':
    os.chdir('..')
sys.path.append(os.getcwd())

from simulation import FluidSimulation
from models import GNNSimulator
from dataset import build_graph
from presentation.utils import create_simulation_video, create_rollout_video

print("✓ Uspešno importovani paketi klasicnog numeričkog WCSPH SPH integratora i GNN modela projekta!")


In [ ]:
# Čista numerička simulacija bez mašinskog učenja
print("Generišem video simulacije (po parametrima iz TestSim.ipynb)...")

# Initialize a 3-body system
sim_test = FluidSimulation(
    num_particles=600,
    gravitational_constant=9.81,
    softening_length=0.015,
    stiffness=1000,
    exponent=7.0,
    viscosity=0.3
)

position_scale=0.5

np.random.seed(42)
sim_test.initialize_random_state(
    position_scale=position_scale,
    velocity_scale=0.0,
    mass_range=(1.0, 1.0),
    start=(1,1)
)

# Simulate
dt = 0.0005
total_time = 5.0
fps = 60
save_every = round((total_time/dt)/(total_time*fps))

history_positions, history_velocities, history_times = sim_test.simulate(
    total_time=total_time,
    dt=dt,
    save_every=save_every
)

# Kreirajmo video čistog SPH modela, ova generacija moze potrajati par sekundi po iteraciji videa.
video_numeric = create_simulation_video(
    sim=sim_test,
    history_positions=history_positions,
    num_particles=sim_test.num_particles,
    position_scale=position_scale,
    fps=fps,
    filename='presentation/nans_wcsph_sim.mp4'
)
display(video_numeric)


## 3. Skupovi podataka i inovativni trening (Dataset generation i Fine-Tuning)

Učenje ovog modela nije radikalno svelo prepoznavanje tečnosti na jedan uski domen usred oskudice podataka, već se zasniva na pre-training/fine-tuning ideji:

1. **Osnovno učenje (Pre-training):** Naš model započinje razumevanje sveta treniranjem na obimnom **DeepMind WaterDrop datasetu**. Time mreža stiče robusno univerzalno razumevanje ograničenja kontejnera i dinamika kapljica.
2. **Proprietary Generisanje Skupa Podataka (Fine-Tuning):** Pomoću *izvornog Python koda i implementiranih integratora za simulacije na ovom kursu*, generisali smo naš **proprietary WCSPH fluid dataset**. Mreža se putem transfernog učenja (Transfer Learning) na finom dezenu navikla na našu varijantu numerike.


## 4. Arhitektura GNN Modela (Encode - Process - Decode)

Kinetički podaci generisani našom SPH metodom posmatraju se kao podaci na grafu pre nego što se ubace u model. Fizičko stanje fluida posmatramo kroz dinamičan *Radius Graph*:
- Svaka čestica fluida i granica bazena je **čvor**. Interakcije fluida u izabranom domenu dejstva sila (radijusu dejstva jezgra $h$) postaju **veze/grane (edges)**.
- **Encoder (Kodiranje):** Prvo projektuje prethodne položaje i brzine čestica u latentni (skriveni) matematički prostor dimenzionalnosti $128$ koristeći klasični Multilayer Perceptron.
- **Process (Procesiranje Pritiska):** Deluje kao serija InteractionNetwork algoritama posvećenih prolasku poruka (Message-Passing arhitekture), koji simuliraju razmenu sila sa susednim partiklima.
- **Decode (Dekodiranje Stanja):** Nakon niza iteracija razmene i konvolucije po partiklu grafa, skriveno grafovsko stanje nazad prevodimo u vektor pritiska, odnosno izvedeno predviđeno **ubrzanje** vektora brzine $\mathbf{a}$.

Dobijen izlaz se provlači redom nazad na **Euler Integrator** za konačno predskazivanje $\mathbf{x}_{t+1}$.


In [ ]:
# Brza inicijalizacija parametara i težina fine-tuned Waterdrop + NANS WCSPH modela
CONFIG_PATH = 'configs/wcsph_transfer.yaml'
CHECKPOINT_PATH = 'checkpoints/best_model_wcsph.pt'

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# loadujemo weights
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

# Kreiranje instance naseg projektovanog modela
model = GNNSimulator.from_config(config).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

broj_parametara = sum(p.numel() for p in model.parameters())
print(f"GNN Arhitektura je inicijalizovana.")
print(f"Veličina modela: {broj_parametara:,} ukupno utreniranih parametara.")


## 5. Rezultati i Vizuelizacija GNN modela (Rollout)

Da bismo testirali efektivnost mašinskog učenja u nelinearnom haosu fluida sa neuređenim parametrima stišljivosti, izvršavamo takozvani **Rollout**. Rollout predstavlja prepuštanje generativnih odluka isključivo mašinskom modulu na vremenski red stotina iteracija. Prilikom obimnog Rollout-a kod postavki od preko 4,000 čestica, standardnim mašinama potrebno je više od 10 minuta. Zato u prezentaciji prikazujemo smanjenu trajektoriju tečnosti (od 1,500 čestica) koja obogaćuje interaktivnost ove radne površine i pruža uvid u proračunsku vizualizaciju iz simulacije!


In [ ]:
# Priprema domena i pre-generisanje Ground Truth numeričke simulacije
# ─── WaterDrop-aligned parameters (same as generate_fluid_dataset.py) ───
WATERDROP_BOUNDS = [[0.1, 0.9], [0.1, 0.9]]
WATERDROP_DT = 0.0025          # Effective frame-to-frame dt
INTEGRATION_DT = 0.0005        # Sub-step dt for WCSPH stability
SAVE_EVERY = 5                 # INTEGRATION_DT * SAVE_EVERY = WATERDROP_DT
SMOOTHING_LENGTH = 0.015       # = connectivity_radius
SEQUENCE_LENGTH = 1000         # Frames per trajectory
POSITION_SCALE = 0.8           # Container size: [0, 0.8], shifted by +0.1 → [0.1, 0.9]
OFFSET = 0.1                   # Coordinate shift to align with WaterDrop bounds
PARTICLE_TYPE_FLUID = 5        # DeepMind's encoding for fluid particles

# Random seed for this visualization (change for different trajectories)
VIZ_SEED = 123
rng = np.random.RandomState(VIZ_SEED)

# Random particle count in [200, 400]
num_particles = rng.randint(2000, 4001)

# Total simulation time to produce SEQUENCE_LENGTH + 1 frames
total_time = SEQUENCE_LENGTH * SAVE_EVERY * INTEGRATION_DT

sim = FluidSimulation(
    num_particles=num_particles,
    gravitational_constant=9.81,
    softening_length=SMOOTHING_LENGTH,
    integrator='symplectic_euler',
    position_scale=POSITION_SCALE,
    rest_density=1000.0,
    stiffness=1000.0,
    exponent=7.0,
    viscosity=0.3,
)

# Randomized initial drop center within margins
margin = 0.15 * POSITION_SCALE
cx = rng.uniform(margin, POSITION_SCALE - margin)
cy = rng.uniform(0.3 * POSITION_SCALE, POSITION_SCALE - margin)

sim.initialize_random_state(
    position_scale=POSITION_SCALE,
    velocity_scale=0.0,
    mass_range=(1.0, 1.0),
    start=(cx, cy),
)

print(f"Generating trajectory: {num_particles} particles, drop center=({cx:.3f}, {cy:.3f})")
positions, _, _ = sim.simulate(
    total_time=total_time,
    dt=INTEGRATION_DT,
    save_every=SAVE_EVERY,
    quiet=False
)

# Trim and shift to WaterDrop coordinate system
positions = (positions[:SEQUENCE_LENGTH + 1] + OFFSET).astype(np.float32)

# Derive ground truth velocities as spatial frame deltas to match DeepMind format
gt_velocities = (positions[1:] - positions[:-1]).astype(np.float32)
gt_positions = positions[1:].astype(np.float32)

particle_type = np.full(num_particles, PARTICLE_TYPE_FLUID, dtype=np.int64)
mass = np.ones(num_particles, dtype=np.float32)

T, N, D = gt_positions.shape
print(f"Trajectory: {T} timesteps, {N} particles, {D}D")
print(f"Position range: [{gt_positions.min():.4f}, {gt_positions.max():.4f}]")

# Build metadata dict matching WaterDrop format
metadata = {
    'bounds': WATERDROP_BOUNDS,
    'default_connectivity_radius': SMOOTHING_LENGTH,
    'dt': WATERDROP_DT,
}
bounds = metadata['bounds']


In [ ]:
from tqdm.notebook import tqdm

print("3. Predikcije GNN Simulatora - Faza Autoregresivnog Rollout-a (učenje iz proslosti)...")

history_window = 5 # 5 sekvencijalnih koraka koje gleda GNN
pred_positions = np.zeros_like(gt_positions[:T])
pred_positions[:history_window + 1] = gt_positions[:history_window + 1].copy()

# Konverzija za Torch Device
particle_type_t = torch.from_numpy(particle_type).to(device)
mass_t = torch.from_numpy(mass).to(device)
dummy_acc_t = torch.zeros(num_particles, 2, dtype=torch.float32, device=device)

vel_history_t = torch.from_numpy(gt_velocities[:history_window]).to(device)
current_pos_t = torch.from_numpy(gt_positions[history_window]).to(device)
current_vel_t = torch.from_numpy(gt_velocities[history_window - 1]).to(device)

# Sekvencijalna Forward inferenca modela!
with torch.no_grad():
    for t in tqdm(range(history_window, T - 1), desc='Stanje Mreže'):
        vel_flat_t = vel_history_t.transpose(0, 1).reshape(num_particles, history_window * 2)
        
        # Oformljuje radijus graf na bazi susedstva čestica preko CPU/GPU-a
        graph = build_graph(
            positions=current_pos_t, velocities=vel_flat_t, particle_type=particle_type_t,
            masses=mass_t, target_acc=dummy_acc_t, connectivity_radius=SMOOTHING_LENGTH, bounds=WATERDROP_BOUNDS
        )
        
        predicted_acc_t = model(graph)
        new_vel_t = current_vel_t + predicted_acc_t
        new_pos_t = current_pos_t + new_vel_t
        
        pred_positions[t + 1] = new_pos_t.cpu().numpy()
        vel_history_t = torch.cat([vel_history_t[1:], new_vel_t.unsqueeze(0)], dim=0)
        current_pos_t = new_pos_t
        current_vel_t = new_vel_t

# Per-step position MSE
rollout_start = history_window + 1
step_mse = np.mean((pred_positions[rollout_start:] - gt_positions[rollout_start:])**2, axis=(1, 2))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(rollout_start, T), step_mse, color='#e74c3c', linewidth=1.5)
ax.set_xlabel('Timestep')
ax.set_ylabel('Position MSE')
ax.set_title('WCSPH Rollout Error Over Time')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Mean rollout MSE: {step_mse.mean():.6f}")
print(f"Final step MSE: {step_mse[-1]:.6f}")


In [ ]:
# Vizualizacija - Uporedni prikaz istinitih i predvidivih položaja čestica (Animacija)
print("Generisem uporedni rollout video...")
video_rollout = create_rollout_video(
    gt_positions=gt_positions,
    pred_positions=pred_positions,
    num_particles=num_particles,
    T=T,
    sim_dt=WATERDROP_DT,
    bounds=WATERDROP_BOUNDS,
    target_fps=60,
    filename='presentation/nans_gnn_rollout.mp4'
)
display(video_rollout)


## 6. Zaključak kursnog projekta

Ovaj uvid u polje simulacije tečnosti ističe moć prožimanja strogih fizičkih zakona predviđenih matematičkom jednačinom sa moćnom generalizacijom dubokih neuronskih mreža.
Izvršenjem **Symplectic Euler** bazirane integracije i finim tuning-om (*Transfer Learning*) DeepMind struktura, kreirana arhitektura uči iteriranje dinamike složenog nelinearnog sveta pri izvesnom broju čestica bez žrtvovanja stabilnosti - otvarajući vrata bržim i skalabilnijim kompjuterskim grafikama sutrašnjice.
